In [ ]:
%pip install -q "monai==1.5.2" nibabel tqdm

In [ ]:
import csv
import json
import math
import random
import time
from pathlib import Path

import nibabel as nib
import numpy as np
import torch
import monai
from tqdm.auto import tqdm
from monai.data import DataLoader, Dataset, pad_list_data_collate
from monai.inferers import sliding_window_inference
from monai.losses import DiceCELoss
from monai.networks.nets import AttentionUnet
from monai.transforms import (
    CastToTyped,
    Compose,
    EnsureChannelFirstd,
    Lambdad,
    LoadImaged,
    RandAdjustContrastd,
    RandAffined,
    RandCropByPosNegLabeld,
    RandFlipd,
    RandGaussianNoised,
    RandScaleIntensityd,
)

print("PyTorch:", torch.__version__, "| MONAI:", monai.__version__)

In [ ]:
DATA_ROOT = Path("data/tumor")
SPLIT_FILE = Path("data/splits.json")
OUTPUT_DIR = Path("outputs/tumor_attention_unet")
IMAGE_SUFFIX = "_image.nii.gz"
LABEL_SUFFIX = "_tumor_label.nii.gz"
TUMOR_LABEL_VALUE = 1

INPUT_REQUIREMENTS = (
    "Each subject folder DATA_ROOT/<subject_id> contains <subject_id>" + IMAGE_SUFFIX + ", a preprocessed "
    "contrast-enhanced T1-weighted MRI volume (RAS orientation, 0.6 mm isotropic voxels, skull-stripped, "
    "N4 bias field corrected, intensities min-max scaled to 0-255), and <subject_id>" + LABEL_SUFFIX + ", "
    "the reference tumor mask in which the contrast-enhancing and necrotic tumor components are labelled "
    "TUMOR_LABEL_VALUE and all other voxels are 0. SPLIT_FILE is a JSON file with the keys train, validation, "
    "and test, each listing subject identifiers."
)

HYPERPARAMETER_TUNING_SEED = 42
FINAL_TRAINING_SEED = 123
SEED = FINAL_TRAINING_SEED

TARGET_SPACING = (0.6, 0.6, 0.6)
NUM_CLASSES = 2

PATCH_SIZE = (128, 128, 128)
POSITIVE_SAMPLING_WEIGHT = 4.0
NEGATIVE_SAMPLING_WEIGHT = 1.0
SAMPLES_PER_VOLUME = 4
VOLUMES_PER_BATCH = 2

CHANNELS = (32, 64, 128, 256, 512)
STRIDES = (2, 2, 2, 2)
DROPOUT = 0.20

LEARNING_RATE = 6e-5
WEIGHT_DECAY = 5e-5
WARMUP_EPOCHS = 5
COSINE_END_EPOCH = 50
MIN_LEARNING_RATE = 1e-6
MAX_EPOCHS = 100

AUG_FLIP_PROB = 0.50
AUG_AFFINE_PROB = 0.45
AUG_ROTATE_RANGE = (0.15, 0.15, 0.15)
AUG_SCALE_RANGE = (0.15, 0.15, 0.15)
AUG_NOISE_PROB = 0.30
AUG_NOISE_MEAN = 0.0
AUG_NOISE_STD = 0.015
AUG_SCALE_INTENSITY_PROB = 0.40
AUG_SCALE_INTENSITY_FACTOR = 0.15
AUG_CONTRAST_PROB = 0.30
AUG_CONTRAST_GAMMA = (0.65, 1.45)

SLIDING_WINDOW_OVERLAP = 0.50
SLIDING_WINDOW_BATCH = 1
VALIDATION_BATCH_SIZE = 1

NUM_WORKERS = 4
RESUME_FROM_LATEST_CHECKPOINT = False

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_split(split_file):
    with open(split_file, "r", encoding="utf-8") as handle:
        split = json.load(handle)
    for key in ("train", "validation", "test"):
        if key not in split:
            raise KeyError(f"The split file must contain the key '{key}'.")
    train_ids, validation_ids, test_ids = set(split["train"]), set(split["validation"]), set(split["test"])
    if train_ids & validation_ids or train_ids & test_ids or validation_ids & test_ids:
        raise ValueError("The training, validation, and test cohorts must not overlap.")
    return split


def check_input_volume(path):
    volume = nib.load(path)
    spacing = tuple(float(value) for value in volume.header.get_zooms()[:3])
    if not np.allclose(spacing, TARGET_SPACING, atol=1e-3):
        raise ValueError(f"Expected {TARGET_SPACING} mm voxels, found {spacing} in {path}")
    orientation = nib.aff2axcodes(volume.affine)
    if orientation != ("R", "A", "S"):
        raise ValueError(f"Expected RAS orientation, found {orientation} in {path}")


def build_file_list(subject_ids, require_label=True):
    files = []
    for subject_id in subject_ids:
        folder = DATA_ROOT / subject_id
        item = {
            "subject_id": subject_id,
            "image": str(folder / f"{subject_id}{IMAGE_SUFFIX}"),
            "label": str(folder / f"{subject_id}{LABEL_SUFFIX}"),
        }
        if not Path(item["image"]).is_file():
            raise FileNotFoundError(item["image"])
        if require_label and not Path(item["label"]).is_file():
            raise FileNotFoundError(item["label"])
        check_input_volume(item["image"])
        files.append(item)
    return files


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(INPUT_REQUIREMENTS)
split = load_split(SPLIT_FILE)
train_files = build_file_list(split["train"])
validation_files = build_file_list(split["validation"])
test_files = build_file_list(split["test"], require_label=False)
print("Training:", len(train_files), "| Validation:", len(validation_files), "| Test:", len(test_files))
print("Device:", device, "| Seed:", SEED)

In [ ]:
def binarize_tumor_label(label):
    if isinstance(label, torch.Tensor):
        return (label == TUMOR_LABEL_VALUE).to(label.dtype)
    return (np.asarray(label) == TUMOR_LABEL_VALUE).astype(np.float32)


train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Lambdad(keys=["label"], func=binarize_tumor_label),
    RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=PATCH_SIZE,
        pos=POSITIVE_SAMPLING_WEIGHT,
        neg=NEGATIVE_SAMPLING_WEIGHT,
        num_samples=SAMPLES_PER_VOLUME,
        image_key="image",
        image_threshold=0,
    ),
    RandFlipd(keys=["image", "label"], prob=AUG_FLIP_PROB, spatial_axis=0),
    RandFlipd(keys=["image", "label"], prob=AUG_FLIP_PROB, spatial_axis=1),
    RandFlipd(keys=["image", "label"], prob=AUG_FLIP_PROB, spatial_axis=2),
    RandAffined(
        keys=["image", "label"],
        prob=AUG_AFFINE_PROB,
        rotate_range=AUG_ROTATE_RANGE,
        scale_range=AUG_SCALE_RANGE,
        mode=("bilinear", "nearest"),
        padding_mode="zeros",
    ),
    RandGaussianNoised(keys=["image"], prob=AUG_NOISE_PROB, mean=AUG_NOISE_MEAN, std=AUG_NOISE_STD),
    RandScaleIntensityd(keys=["image"], factors=AUG_SCALE_INTENSITY_FACTOR, prob=AUG_SCALE_INTENSITY_PROB),
    RandAdjustContrastd(keys=["image"], prob=AUG_CONTRAST_PROB, gamma=AUG_CONTRAST_GAMMA),
    CastToTyped(keys=["image", "label"], dtype=(np.float32, np.int64)),
])

validation_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Lambdad(keys=["label"], func=binarize_tumor_label),
    CastToTyped(keys=["image", "label"], dtype=(np.float32, np.int64)),
])

inference_transforms = Compose([
    LoadImaged(keys=["image"]),
    EnsureChannelFirstd(keys=["image"]),
    CastToTyped(keys=["image"], dtype=np.float32),
])

loader_options = {"num_workers": NUM_WORKERS, "pin_memory": device.type == "cuda", "collate_fn": pad_list_data_collate}

train_loader = DataLoader(
    Dataset(data=train_files, transform=train_transforms),
    batch_size=VOLUMES_PER_BATCH,
    shuffle=True,
    **loader_options,
)
validation_loader = DataLoader(
    Dataset(data=validation_files, transform=validation_transforms),
    batch_size=VALIDATION_BATCH_SIZE,
    shuffle=False,
    **loader_options,
)
print("Training batches per epoch:", len(train_loader), "| Validation volumes:", len(validation_loader))

In [ ]:
model = AttentionUnet(
    spatial_dims=3,
    in_channels=1,
    out_channels=NUM_CLASSES,
    channels=CHANNELS,
    strides=STRIDES,
    dropout=DROPOUT,
).to(device)
trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print(f"Attention U-Net trainable parameters: {trainable_parameters:,}")

loss_function = DiceCELoss(to_onehot_y=True, softmax=True, include_background=False)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


def learning_rate_for_epoch(epoch):
    if epoch <= WARMUP_EPOCHS:
        return LEARNING_RATE * epoch / max(WARMUP_EPOCHS, 1)
    if epoch <= COSINE_END_EPOCH:
        progress = (epoch - WARMUP_EPOCHS) / max(COSINE_END_EPOCH - WARMUP_EPOCHS, 1)
        return MIN_LEARNING_RATE + (LEARNING_RATE - MIN_LEARNING_RATE) * 0.5 * (1.0 + math.cos(math.pi * progress))
    return MIN_LEARNING_RATE

In [ ]:
def nan_mean(values):
    finite = [float(value) for value in values if value is not None and math.isfinite(float(value))]
    return float(np.mean(finite)) if finite else float("nan")


def case_metrics(logits, labels):
    prediction = torch.argmax(logits, dim=1)[0] == 1
    reference = labels[0, 0] == 1
    tp = float(torch.logical_and(prediction, reference).sum())
    fp = float(torch.logical_and(prediction, ~reference).sum())
    fn = float(torch.logical_and(~prediction, reference).sum())
    return {
        "dice": 2.0 * tp / (2.0 * tp + fp + fn) if (2.0 * tp + fp + fn) > 0 else float("nan"),
        "precision": tp / (tp + fp) if (tp + fp) > 0 else float("nan"),
        "recall": tp / (tp + fn) if (tp + fn) > 0 else float("nan"),
    }


METRIC_KEYS = ["dice", "precision", "recall"]


def summarize_cases(rows):
    return {key: nan_mean([row[key] for row in rows]) for key in METRIC_KEYS}


def train_one_epoch(epoch):
    model.train()
    losses, patch_rows = [], []
    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{MAX_EPOCHS} training", leave=False):
        inputs = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)
        loss = loss_function(logits, labels)
        loss.backward()
        optimizer.step()
        losses.append(float(loss.detach().item()))
        with torch.no_grad():
            for index in range(logits.shape[0]):
                patch_rows.append(case_metrics(logits[index:index + 1], labels[index:index + 1]))
    summary = summarize_cases(patch_rows)
    summary["loss"] = nan_mean(losses)
    return summary


def validate(epoch):
    model.eval()
    losses, case_rows = [], []
    with torch.no_grad():
        for batch in tqdm(validation_loader, desc=f"Epoch {epoch}/{MAX_EPOCHS} validation", leave=False):
            inputs = batch["image"].to(device)
            labels = batch["label"].to(device)
            logits = sliding_window_inference(
                inputs,
                roi_size=PATCH_SIZE,
                sw_batch_size=SLIDING_WINDOW_BATCH,
                predictor=model,
                overlap=SLIDING_WINDOW_OVERLAP,
            )
            losses.append(float(loss_function(logits, labels).item()))
            row = case_metrics(logits, labels)
            row["subject_id"] = batch["subject_id"][0]
            case_rows.append(row)
    summary = summarize_cases(case_rows)
    summary["loss"] = nan_mean(losses)
    return summary, case_rows

In [ ]:
HISTORY_FILE = OUTPUT_DIR / "training_history.csv"
VALIDATION_CASES_FILE = OUTPUT_DIR / "validation_cases.csv"
LATEST_CHECKPOINT_FILE = OUTPUT_DIR / "latest_checkpoint.pt"
BEST_MODEL_FILE = OUTPUT_DIR / "best_model.pt"
SUMMARY_KEYS = ["loss"] + METRIC_KEYS
HISTORY_FIELDS = ["epoch", "learning_rate"] + [f"train_{key}" for key in SUMMARY_KEYS] + [f"validation_{key}" for key in SUMMARY_KEYS]
CASE_FIELDS = ["epoch", "subject_id"] + METRIC_KEYS


def append_rows(path, fields, rows):
    write_header = not path.exists()
    with open(path, "a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        if write_header:
            writer.writeheader()
        writer.writerows(rows)


start_epoch = 1
best_validation_dice = -1.0
best_epoch = None
if RESUME_FROM_LATEST_CHECKPOINT and LATEST_CHECKPOINT_FILE.is_file():
    state = torch.load(LATEST_CHECKPOINT_FILE, map_location=device)
    model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    start_epoch = int(state["epoch"]) + 1
    best_validation_dice = float(state["best_validation_dice"])
    best_epoch = state["best_epoch"]
    print("Resuming from epoch", start_epoch)

for epoch in range(start_epoch, MAX_EPOCHS + 1):
    started = time.time()
    learning_rate = learning_rate_for_epoch(epoch)
    for group in optimizer.param_groups:
        group["lr"] = learning_rate

    train_summary = train_one_epoch(epoch)
    validation_summary, validation_cases = validate(epoch)

    if validation_summary["dice"] > best_validation_dice:
        best_validation_dice = validation_summary["dice"]
        best_epoch = epoch
        torch.save(model.state_dict(), BEST_MODEL_FILE)

    history_row = {"epoch": epoch, "learning_rate": learning_rate}
    history_row.update({f"train_{key}": train_summary[key] for key in SUMMARY_KEYS})
    history_row.update({f"validation_{key}": validation_summary[key] for key in SUMMARY_KEYS})
    append_rows(HISTORY_FILE, HISTORY_FIELDS, [history_row])
    append_rows(VALIDATION_CASES_FILE, CASE_FIELDS, [{"epoch": epoch, **row} for row in validation_cases])

    torch.save(
        {
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "best_validation_dice": best_validation_dice,
            "best_epoch": best_epoch,
        },
        LATEST_CHECKPOINT_FILE,
    )
    print(
        f"Epoch {epoch}/{MAX_EPOCHS} | lr {learning_rate:.2e} | train loss {train_summary['loss']:.4f} | "
        f"validation loss {validation_summary['loss']:.4f} | validation DSC {validation_summary['dice']:.4f} | "
        f"{time.time() - started:.0f} s"
    )

print("Selected checkpoint: epoch", best_epoch, "| validation DSC", best_validation_dice)

In [ ]:
PREDICTION_DIR = OUTPUT_DIR / "test_predictions"
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
model.load_state_dict(torch.load(BEST_MODEL_FILE, map_location=device))
model.eval()
with torch.no_grad():
    for item in tqdm(test_files, desc="Test cohort inference"):
        data = inference_transforms({"image": item["image"]})
        inputs = data["image"].unsqueeze(0).to(device)
        logits = sliding_window_inference(
            inputs,
            roi_size=PATCH_SIZE,
            sw_batch_size=SLIDING_WINDOW_BATCH,
            predictor=model,
            overlap=SLIDING_WINDOW_OVERLAP,
        )
        prediction = torch.argmax(logits, dim=1)[0].cpu().numpy().astype(np.uint8)
        reference = nib.load(item["image"])
        nib.save(nib.Nifti1Image(prediction, reference.affine), str(PREDICTION_DIR / f"{item['subject_id']}_prediction.nii.gz"))
print("Test cohort predictions saved to", PREDICTION_DIR)